# Flux reconstruction, error indicators and adaptive decisions

This lesson starts with user-written UFL local/global equations. It then defines a physical coefficient record, reconstructs an H(div) flux, evaluates an estimator and chooses cells for refinement. Numerical algorithms remain in the package; no physical-model constructor selects the equations.

The primary case is $p=\sin(2\pi x)\sin(2\pi y)$, $K=I$, $f=8\pi^2p$, with homogeneous Dirichlet pressure on the unit square. The formulation follows [Harder, Paredes and Valentin (2013)](https://doi.org/10.1016/j.jcp.2013.03.019). Flux recovery and the four-term estimator follow [Barrenechea, Martins, Pereira and Valentin (2026)](https://doi.org/10.1137/24M1673073).

Install the native UFL backend as explained in the installation guide. The workflow is **meshes → local forms → global forms → assemble → solve → recovery → indicators → marking**.

In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download checksum-verified support files; extraction does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/44c48b361b54fb06afffe969785950024502d0eaff280d0fe33adcc41aa6532d/reconstruction_and_indicators-companion.zip"
COMPANION_SHA256 = "44c48b361b54fb06afffe969785950024502d0eaff280d0fe33adcc41aa6532d"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Prepare only this notebook's declared inputs, without executing its equations.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("darcy/reconstruction_and_indicators.ipynb", directory=ROOT)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ufl
from pymhm import (
    Equation, FaceSpace, LocalContext, MeshHierarchy, SkeletonSpace, TriangleMesh,
    assemble, bind_interface, bind_problem, columns, solve,
)
from pymhm.postprocessing.solutions import DarcySolution
from pymhm.recovery.moments import reconstruct_darcy_moments
from pymhm.estimators.darcy import estimate_darcy_error
from pymhm.adaptivity.darcy import mark_dorfler
from threadpoolctl import threadpool_limits


def exact_pressure(points: np.ndarray) -> np.ndarray:
    """Analytical pressure; it vanishes on every exterior edge."""
    return np.sin(2 * np.pi * points[:, 0]) * np.sin(2 * np.pi * points[:, 1])


def exact_gradient(points: np.ndarray) -> np.ndarray:
    """Independently differentiated physical pressure gradient."""
    x, y = 2 * np.pi * points.T
    return 2 * np.pi * np.column_stack((np.cos(x) * np.sin(y), np.sin(x) * np.cos(y)))


def source(points: np.ndarray) -> np.ndarray:
    """Negative Laplacian computed independently of the discrete operator."""
    return 8 * np.pi**2 * exact_pressure(points)

## 1. Choose meshes and approximation spaces

For reconstruction degree $m=2$ and trace degree $\ell=1$, choose local degree $k=3$. This satisfies the two-dimensional restriction $k\ge\ell+2$ and $\ell\le m\le k$. The estimator additionally needs convex macrotriangles, a globally conforming fine partition, identity diffusion and homogeneous Dirichlet data. These are mathematical restrictions, not conventions inferred by the software.

In [ ]:
macro = TriangleMesh.unit_square(2)
local_meshes = tuple(macro.submesh(cell, 2) for cell in range(len(macro.cells)))
hierarchy = MeshHierarchy(macro, local_meshes)
skeleton = SkeletonSpace(macro, tuple(FaceSpace.uniform(1) for _ in macro.faces))
interface = bind_interface(skeleton, convention="normal")

## 2. Declare local equations

$$
\begin{aligned}
(\nabla p_T,\nabla v)_T+\langle\lambda_T,v\rangle_{\partial T}&=(f,v)_T,\\
-\langle p_T,\mu_T\rangle_{\partial T}&=g_T(\mu_T).
\end{aligned}
$$

The local volume form has a constant kernel. Its physical volume moment fixes the local complement and retains one mean per macrocell. The interface binding supplies normal incidence and numbering; the minus sign in the global test pairing is part of the formulation.

In [ ]:
def local_equations(local: LocalContext):
    """Declare primal diffusion, independent trace tests and physical mean."""
    space = local.native_space(degree=3)
    p, v = ufl.TrialFunction(space.space), ufl.TestFunction(space.space)
    x = ufl.SpatialCoordinate(space.mesh)
    dx = ufl.Measure("dx", domain=space.mesh, metadata={"quadrature_degree": 12})
    forcing = 8 * np.pi**2 * ufl.sin(2 * np.pi * x[0]) * ufl.sin(2 * np.pi * x[1])
    local.field("pressure", space)
    return local.equations(
        a=ufl.inner(ufl.grad(p), ufl.grad(v)) * dx,
        L=forcing * v * dx,
        b=local.trace_pairings(lambda phi, ds: phi * v * ds),
        c=local.trace_pairings(lambda phi, ds: -phi * p * ds, axis="rows"),
        kernel=np.ones((space.size, 1)), moments=columns(v * dx),
    )

## 3. Declare global boundary moments

$$
-\sum_T\langle p_T,\mu_T\rangle_{\partial T}
=-\langle p_D,\mu\rangle_{\partial\Omega},\qquad p_D=0.
$$

The Dirichlet load is therefore zero. Its sign is kept explicit so that changing the boundary data does not change the formulation accidentally.

In [ ]:
def global_equation(global_context):
    """Supply the weak Dirichlet moments in the bound interface layout."""
    boundary, fixed = global_context.boundary_data(0.0, order=8)
    assert not fixed
    return Equation(0, global_context.trace_load(-boundary))


problem = bind_problem(hierarchy, interface, local_equations,
                       global_equation=global_equation, retained=1)

## 4. Assemble, solve and name the physical result

Named fields preserve the native executed coefficient basis. `portable_coefficients` converts through its stored mapping; reshaping native vectors would not be equivalent. `DarcySolution` below is a data record for established reconstruction and estimator operations. Constructing it performs no PDE solve.

In [ ]:
with threadpool_limits(1):
    system = assemble(problem)
    coefficients = solve(system)
pressure_fields = coefficients.field("pressure")
solution = DarcySolution(
    skeleton=skeleton, local_meshes=local_meshes,
    pressure=tuple(field.portable_coefficients for field in pressure_fields),
    flux=(), hybrid=coefficients, formulation="primal", permeability=1.0,
    source=source, degree=3, quadrature_order=10,
)
print({"original_equation_residual": coefficients.raw_residual,
       "pressure_L2": solution.l2_error(exact_pressure, order=12)})

## 5. Reconstruct a flux and distinguish its conservation tests

The canonical RT moment reconstruction preserves boundary normal moments from the skeleton, averages interior normal moments and preserves raw interior vector moments. It is not the energy-minimizing RT0 equilibration. Its divergence balance is against continuous macro-local tests, not individual discontinuous fine-cell constants.

$$
\begin{aligned}
\langle q_h\cdot n,\phi\rangle_E&=\text{declared normal moments},\\
(q_h,\psi)_t&=(-\nabla p_h,\psi)_t.
\end{aligned}
$$

In [ ]:
recovered = reconstruct_darcy_moments(solution, degree=2, quadrature_order=10)
normal_defect = max(np.max(abs(row)) for row in recovered.normal_flux_residuals())
continuous_defect = max(np.max(abs(row)) for row in recovered.continuous_moment_residuals())
fine_defect = max(np.max(abs(row)) for row in recovered.fine_conservation_residuals())
assert normal_defect < 1e-10
assert continuous_defect < 1e-10
print({"normal_moment_defect": normal_defect,
       "continuous_test_balance_defect": continuous_defect,
       "fine_cell_balance_defect_not_imposed": fine_defect,
       "recovered_flux_L2": recovered.flux_l2_error(lambda x: -exact_gradient(x), order=12)})

## 6. Evaluate the four-term indicator

$$
\eta^2=\sum_T\big[(\eta_{1,T}+\eta_{3,T}+\eta_{\mathrm{osc},T})^2
+\eta_{2,T}^2\big].
$$

The contributions measure flux defect, potential nonconformity, divergence-projection defect and source oscillation. The recovered potential is a separate continuous field. For this unit-diffusion case, compare the estimator with the independently integrated broken energy error; numerical quadrature does not provide an interval-certified bound.

In [ ]:
estimate = estimate_darcy_error(solution, homogeneous_dirichlet=True,
                                degree=2, quadrature_order=10)
energy_error = estimate.energy_error(exact_gradient, order=12)
assert max(estimate.equilibrium_defect) < 1e-10
assert estimate.total >= energy_error
print({"energy_error": energy_error, "estimator": estimate.total,
       "effectivity": estimate.total / energy_error})

## 7. Plot physical fields with the actual macro mesh

Each field panel has its own color scale. Numerical values and errors are sampled independently in each local mesh, preserving interface sides.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), layout="constrained")
all_exact, all_values, panels = [], [], []
for fine, field in zip(local_meshes, pressure_fields, strict=True):
    points = fine.points[fine.cells].mean(axis=1)
    exact, values = exact_pressure(points), field.evaluate(points)
    panels.append((fine, exact, values))
    all_exact.extend(exact)
    all_values.extend(values)
common = max(np.max(np.abs(all_exact)), np.max(np.abs(all_values)))
error_max = max(np.max(abs(values - exact)) for _, exact, values in panels)
for index, (ax, title) in enumerate(zip(axes, ("Analytical pressure", "MHM pressure", "Pressure error"), strict=True)):
    for fine, exact, values in panels:
        value = (exact, values, values - exact)[index]
        limit = error_max if index == 2 else common
        artist = ax.tripcolor(*fine.points.T, fine.cells, facecolors=value,
                             vmin=-limit, vmax=limit, cmap="coolwarm")
    for edge in macro.faces:
        ax.plot(*macro.points[edge].T, color="black", lw=0.5, alpha=0.7)
    ax.set(title=title, xlabel="x", ylabel="y", aspect="equal")
    fig.colorbar(artist, ax=ax)
plt.show()

## 8. Mark cells; choose the scale you want to improve

Dörfler marking selects a minimal sorted collection carrying the prescribed fraction of the squared indicator:

$$
\sum_{T\in\mathcal M}\eta_T^2\ge\theta\sum_T\eta_T^2.
$$

Marking is distinct from refining. Before changing a mesh, choose whether the dominant error comes from the macro partition, local approximation, or trace resolution. Refinement ancestry must preserve material and boundary markers. Rebuild the equations on the new spaces and compare error against work; a general adaptive sequence has no predetermined uniform-grid rate.

In [ ]:
marked = mark_dorfler(estimate.local_squared, theta=0.5)
assert estimate.local_squared[marked].sum() >= 0.5 * estimate.local_squared.sum()
fig, ax = plt.subplots(figsize=(5, 4), layout="constrained")
artist = ax.tripcolor(*macro.points.T, macro.cells, facecolors=estimate.local_squared,
                     edgecolors="black", cmap="viridis")
centers = macro.points[macro.cells[marked]].mean(axis=1)
ax.scatter(*centers.T, marker="x", color="white", s=60, label="Dörfler marked")
ax.set(xlabel="x", ylabel="y", title="Squared macro indicators", aspect="equal")
ax.legend()
fig.colorbar(artist, ax=ax)
plt.show()
print({"marked_macro_cells": np.flatnonzero(marked).tolist(), "marked_fraction":
       float(estimate.local_squared[marked].sum() / estimate.local_squared.sum())})

## 9. Refine, rebuild the declared equations and compare work

This loop applies the same user-written local and global forms to successively marked macro meshes. The shared refinement owner makes a conforming triangulation and returns ancestry; the two local edge subdivisions keep the fine meshes globally conforming. The plot compares the actual energy error and estimator with global trace unknowns. It does not fit a uniform-grid convergence exponent to an adaptive path.

In [ ]:
from pymhm.meshes.refinement import refine_triangles

adaptive_rows = [{"macro_cells": len(macro.cells), "trace_dofs": skeleton.size,
                  "energy_error": energy_error, "estimator": estimate.total}]
for _ in range(3):
    marked = mark_dorfler(estimate.local_squared, theta=0.5)
    refinement = refine_triangles(macro, marked)
    macro = refinement.mesh
    local_meshes = tuple(macro.submesh(i, 2) for i in range(len(macro.cells)))
    hierarchy = MeshHierarchy(macro, local_meshes)
    skeleton = SkeletonSpace(macro, tuple(FaceSpace.uniform(1) for _ in macro.faces))
    interface = bind_interface(skeleton, convention="normal")
    problem = bind_problem(hierarchy, interface, local_equations,
                           global_equation=global_equation, retained=1)
    with threadpool_limits(1):
        system = assemble(problem)
        coefficients = solve(system)
    pressure_fields = coefficients.field("pressure")
    solution = DarcySolution(
        skeleton=skeleton, local_meshes=local_meshes,
        pressure=tuple(field.portable_coefficients for field in pressure_fields),
        flux=(), hybrid=coefficients, formulation="primal", permeability=1.0,
        source=source, degree=3, quadrature_order=10,
    )
    estimate = estimate_darcy_error(solution, homogeneous_dirichlet=True,
                                    degree=2, quadrature_order=10)
    energy_error = estimate.energy_error(exact_gradient, order=12)
    assert max(estimate.equilibrium_defect) < 1e-10
    assert estimate.total >= energy_error
    assert energy_error < adaptive_rows[-1]["energy_error"]
    adaptive_rows.append({"macro_cells": len(macro.cells), "trace_dofs": skeleton.size,
                          "energy_error": energy_error, "estimator": estimate.total})
for row in adaptive_rows:
    print(row)
fig, ax = plt.subplots(figsize=(6, 4), layout="constrained")
work = [row["trace_dofs"] for row in adaptive_rows]
for name in ("energy_error", "estimator"):
    ax.loglog(work, [row[name] for row in adaptive_rows], "o-", label=name.replace("_", " "))
ax.set(xlabel="Global trace unknowns", ylabel="Physical energy quantity",
       title="Adaptive error and work")
ax.grid(True, which="both", alpha=0.25)
ax.legend()
plt.show()

## 10. Recover a conforming potential and a separately equilibrated RT0 flux

The Oswald potential averages incident finite-element nodal traces in the global continuous P3 space, then imposes the declared homogeneous boundary data. Its global coefficient vector supplies a single value at every shared node. This is a different field from the original broken pressure.

For strict fine-cell balance, instead solve the weighted RT0 minimization with the same physical source and a **degree-zero aligned** skeletal trace. Reassemble those newly declared spaces first; do not pass incompatible P1 trace data to the RT0 operator. The minimization is an available reconstruction, not an assertion of equivalence to the published RT2 moment operator.

In [ ]:
from pymhm.estimators.darcy import recover_potential
from pymhm.recovery.equilibrated import equilibrate_flux
from pymhm.fem.scalar.triangle import nodal_space

potential = recover_potential(solution, homogeneous_dirichlet=True)
_, potential_points = nodal_space(potential.mesh, potential.degree)
boundary_nodes = np.any(np.isclose(potential_points, 0.0) | np.isclose(potential_points, 1.0), axis=1)
assert np.max(abs(potential.values[boundary_nodes])) == 0.0
print({"conforming_potential_L2": potential.l2_error(exact_pressure, order=12),
       "potential_boundary_defect": float(np.max(abs(potential.values[boundary_nodes])))})

rt0_skeleton = SkeletonSpace(macro)
rt0_interface = bind_interface(rt0_skeleton, convention="normal")
rt0_problem = bind_problem(hierarchy, rt0_interface, local_equations,
                          global_equation=global_equation, retained=1)
with threadpool_limits(1):
    rt0_system = assemble(rt0_problem)
    rt0_coefficients = solve(rt0_system)
rt0_solution = DarcySolution(
    skeleton=rt0_skeleton, local_meshes=local_meshes,
    pressure=tuple(field.portable_coefficients for field in rt0_coefficients.field("pressure")),
    flux=(), hybrid=rt0_coefficients, formulation="primal", permeability=1.0,
    source=source, degree=3, quadrature_order=10,
)
equilibrated = equilibrate_flux(rt0_solution)
fine_balance = max(np.max(abs(row)) for row in equilibrated.conservation_residuals())
assert fine_balance < 1e-10
print({"equilibrated_fine_cell_balance": fine_balance,
       "equilibrated_Darcy_flux_L2": equilibrated.l2_error(lambda x: -exact_gradient(x), order=12)})

## References and extensions

- Harder, Paredes and Valentin (2013), *A family of Multiscale Hybrid-Mixed finite element methods for the Darcy equation with rough coefficients*, [DOI](https://doi.org/10.1016/j.jcp.2013.03.019).
- Barrenechea, Martins, Pereira and Valentin (2026), [DOI](https://doi.org/10.1137/24M1673073).

For a declared SPD coefficient, use the energy-normalized estimator and supply certified ellipticity bounds when required. RT0 energy-minimizing equilibration instead imposes fine-cell averages and needs degree-zero trace segments aligned with fine boundary faces. Flow and elasticity indicators have their own hypotheses. The corresponding tutorials explain these contracts rather than silently applying the scalar indicator to other equations.